Import library

In [53]:
import numpy as np 
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

Read the dataset

In [54]:
df = pd.read_csv('Datasets/covid_toy.csv')
df.sample(5)

,age,gender,fever,cough,city,has_covid
2,42,Male,101.0,Mild,Delhi,No
90,59,Female,99.0,Strong,Delhi,No
58,23,Male,98.0,Strong,Mumbai,Yes
66,51,Male,104.0,Mild,Kolkata,No
71,75,Female,104.0,Strong,Delhi,No


Check the different values present in cit column.

In [55]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

Find the total number of null/missing values.

In [56]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

Train - test split.

In [57]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df.drop('has_covid', axis=1), df['has_covid'], test_size=0.2, random_state=42)

Print sample values.

In [58]:
x_train.sample(5)

,age,gender,fever,cough,city
2,42,Male,101.0,Mild,Delhi
74,34,Female,104.0,Strong,Delhi
96,51,Female,101.0,Strong,Kolkata
8,19,Female,100.0,Strong,Bangalore
64,42,Male,104.0,Mild,Mumbai


Tranforming the data simply (without column transform): -

Fill the missing values in fever column by using simple imputer which learn stastics (mean, median etc) from the training data abd apply to train and test both.

In [59]:
si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])
x_test_fever = si.transform(x_test[['fever']])
x_train_fever.shape, x_test_fever.shape

((80, 1), (20, 1))

OHE on cough column.

In [60]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])

x_test_cough = oe.transform(x_test[['cough']])

x_train_cough.shape

(80, 1)

Convert the data column of age into numpy array and print the shape of array. 

In [61]:
x_train_age = x_train[['age']].values
x_test_age = x_test[['age']].values
x_train_age.shape, x_test_age.shape

((80, 1), (20, 1))

OHE on gender and city column.

In [62]:
ohe = OneHotEncoder(drop='first', sparse_output=False)

x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city  = ohe.transform(x_test[['gender','city']])
x_train_gender_city.shape, x_test_gender_city.shape 

((80, 4), (20, 4))

Join all the part of the original dataset intoa single numpy array.

In [63]:
x_train_transformed = np.hstack([x_train_age.reshape(-1, 1), x_train_fever.reshape(-1, 1), x_train_gender_city, x_train_cough])
x_test_transformed = np.hstack([x_test_age.reshape(-1, 1), x_test_fever.reshape(-1, 1), x_test_gender_city, x_test_cough])
x_train_transformed.shape, x_test_transformed.shape

# # Get column names dynamically
# col_names = ['age', 'fever'] + list(ohe.get_feature_names_out(['gender', 'city'])) + list(oe.get_feature_names_out(['cough']))

# # Convert to DataFrames
# df_train = pd.DataFrame(x_train_transformed, columns=col_names)
# df_test = pd.DataFrame(x_test_transformed, columns=col_names)

# print(df_train.head())
# print(df_train.shape, df_test.shape)

((80, 7), (20, 7))

Now doing the same long process but with the help of Column tranformer.

In [64]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers=[
        ('tnf1', SimpleImputer(),['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
        ('tnf3', OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
        ],remainder='passthrough')
transformer.fit_transform(x_train).shape, transformer.transform(x_test).shape

((80, 7), (20, 7))